# DreamBooth Train Pix2Pix Fine-Tuning for Rumah Gadang

Fine-tune langsung InstructPix2Pix memakai pasangan `input_image -> edited_image` dari dataset generator `generate_instructpix2pix_finetune_images.ipynb`.


In [ ]:
from __future__ import annotations

import os
import sys
import re
import json
import random
import itertools
import pickle
import shutil
from pathlib import Path
from datetime import datetime


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name, "").strip()
    return int(value) if value else default


def env_float(name: str, default: float) -> float:
    value = os.environ.get(name, "").strip()
    return float(value) if value else default


def env_str(name: str, default: str) -> str:
    value = os.environ.get(name, "").strip()
    return value if value else default


def slugify(value: str) -> str:
    value = value.replace(".", "p").replace("-", "m")
    return re.sub(r"[^A-Za-z0-9_]+", "_", value).strip("_")


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESOLUTION = env_int("GADANG_RESOLUTION", 512)
TRAIN_PAIR_COUNT = env_int("GADANG_TRAIN_PIX2PIX_PAIRS", 50)
MAX_TRAIN_STEPS = env_int("GADANG_MAX_TRAIN_STEPS", 2000)
CHECKPOINT_EVERY = env_int("GADANG_CHECKPOINT_EVERY", 200)
LEARNING_RATE = env_float("GADANG_LEARNING_RATE", 1e-4)
TRAIN_BATCH_SIZE = env_int("GADANG_TRAIN_BATCH_SIZE", 1)
GRADIENT_ACCUMULATION_STEPS = env_int("GADANG_GRADIENT_ACCUMULATION_STEPS", 4)
MIXED_PRECISION = env_str("GADANG_MIXED_PRECISION", "fp16")
EVAL_PAIR_COUNT = env_int("GADANG_EVAL_PAIR_COUNT", 4)
EVAL_NUM_INFERENCE_STEPS = env_int("GADANG_EVAL_NUM_INFERENCE_STEPS", 30)
EVAL_GUIDANCE_SCALE = env_float("GADANG_EVAL_GUIDANCE_SCALE", 7.5)
EVAL_IMAGE_GUIDANCE_SCALE = env_float("GADANG_EVAL_IMAGE_GUIDANCE_SCALE", 1.5)
RUN_EVALUATION = env_int("GADANG_RUN_EVALUATION", 1) == 1
RUN_METRICS = env_int("GADANG_RUN_METRICS", 1) == 1
GENERATOR_FILTER = env_str("GADANG_TRAIN_PIX2PIX_GENERATORS", "all")
INSTRUCT_PIX2PIX_CKPT = Path(env_str("GADANG_INSTRUCT_PIX2PIX_CKPT", str(PROJECT_ROOT / "models" / "instruct-pix2pix-00-22000.ckpt")))
RUN_STAMP = env_str("GADANG_RUN_STAMP", datetime.now().strftime("%Y%m%d_%H%M%S"))
SEED = env_int("GADANG_SEED", 100)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")


def find_latest_generated_metadata() -> Path:
    candidates = sorted(PROJECT_ROOT.glob("outputs/*/generated_instructpix2pix_dataset/metadata.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        return candidates[0]
    return PROJECT_ROOT / "outputs" / "missing_generated_instructpix2pix_dataset" / "metadata.jsonl"

DATASET_METADATA_PATH = Path(env_str("GADANG_TRAIN_PIX2PIX_METADATA", str(find_latest_generated_metadata())))


In [ ]:

DREAMBOOTH_SCOPE_LABEL = env_str("GADANG_DREAMBOOTH_SCOPE", env_str("GADANG_LORA_RANK", "8"))
lr_slug = slugify(f"{LEARNING_RATE:.0e}") if LEARNING_RATE < 0.001 else slugify(str(LEARNING_RATE))
DEFAULT_RUN_NAME = f"train_pix2pix_dreambooth_pairs{TRAIN_PAIR_COUNT}_lr{lr_slug}_scope{slugify(str(DREAMBOOTH_SCOPE_LABEL))}_steps{MAX_TRAIN_STEPS}"
RUN_NAME = slugify(env_str("GADANG_RUN_NAME", DEFAULT_RUN_NAME))
METHOD = "train_pix2pix_dreambooth"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME
PAIR_DATASET_DIR = OUTPUT_DIR / "train_pix2pix_pairs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
EVALUATION_DIR = OUTPUT_DIR / "evaluation"
METRICS_DIR = OUTPUT_DIR / "metrics"
LOG_DIR = OUTPUT_DIR / "logs"
NOTEBOOK_OUTPUT_DIR = OUTPUT_DIR / "notebook"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"
ARTIFACT_LOG_PATH = OUTPUT_DIR / "artifact_log.json"

for directory in [OUTPUT_DIR, PAIR_DATASET_DIR, CHECKPOINT_DIR, EVALUATION_DIR, METRICS_DIR, LOG_DIR, NOTEBOOK_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ROOT_LOG_PATH = PROJECT_ROOT / "output.log"
RUN_LOG_PATH = LOG_DIR / "run_output.log"
NOTEBOOK_WRITES_ROOT_LOG = env_int("GADANG_NOTEBOOK_WRITES_ROOT_LOG", 0) == 1

class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()
    def flush(self):
        for stream in self.streams:
            stream.flush()

_run_log_file = RUN_LOG_PATH.open("w", encoding="utf-8")
if NOTEBOOK_WRITES_ROOT_LOG:
    _root_log_file = ROOT_LOG_PATH.open("w", encoding="utf-8")
    sys.stdout = Tee(sys.__stdout__, _root_log_file, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _root_log_file, _run_log_file)
else:
    sys.stdout = Tee(sys.__stdout__, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _run_log_file)

RUN_CONFIG = {
    "method": METHOD,
    "run_name": RUN_NAME,
    "run_stamp": RUN_STAMP,
    "dataset_metadata_path": str(DATASET_METADATA_PATH),
    "train_pair_count": TRAIN_PAIR_COUNT,
    "generator_filter": GENERATOR_FILTER,
    "max_train_steps": MAX_TRAIN_STEPS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "resolution": RESOLUTION,
    "seed": SEED,
    "run_evaluation": RUN_EVALUATION,
    "run_metrics": RUN_METRICS,
    "eval_pair_count": EVAL_PAIR_COUNT,
    "instruct_pix2pix_checkpoint": str(INSTRUCT_PIX2PIX_CKPT),
}
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")


def log_event(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}", flush=True)


def log_section(title: str) -> None:
    log_event("=" * 72)
    log_event(title)
    log_event("=" * 72)

log_section("Notebook initialized")
log_event(f"Project root: {PROJECT_ROOT}")
log_event(f"Dataset metadata: {DATASET_METADATA_PATH}")
log_event(f"InstructPix2Pix checkpoint: {INSTRUCT_PIX2PIX_CKPT}")
log_event(f"Output dir: {OUTPUT_DIR}")
print(json.dumps(RUN_CONFIG, indent=2), flush=True)
assert DATASET_METADATA_PATH.exists(), f"Generated pair metadata not found: {DATASET_METADATA_PATH}"
assert INSTRUCT_PIX2PIX_CKPT.exists(), f"InstructPix2Pix checkpoint not found: {INSTRUCT_PIX2PIX_CKPT}"


## Load Generated Pairs

In [ ]:
from PIL import Image, ImageOps
import pandas as pd
from tqdm.auto import tqdm


def resize_square(image: Image.Image, size: int = RESOLUTION) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    crop_size = min(width, height)
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    return image.resize((size, size), Image.Resampling.LANCZOS)


def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

records = [r for r in load_jsonl(DATASET_METADATA_PATH) if r.get("status") == "ok" and r.get("edited_image")]
if GENERATOR_FILTER.strip().lower() != "all":
    allowed = {item.strip() for item in GENERATOR_FILTER.split() if item.strip()}
    records = [r for r in records if r.get("generator_model") in allowed]
assert records, "No successful generated pairs found after filtering."

rng = random.Random(SEED)
rng.shuffle(records)
selected_records = records[:TRAIN_PAIR_COUNT]
assert len(selected_records) == TRAIN_PAIR_COUNT, f"Need {TRAIN_PAIR_COUNT} generated pairs, found {len(selected_records)}."
eval_records = selected_records[:min(EVAL_PAIR_COUNT, len(selected_records))]

prepared_records = []
for idx, record in enumerate(tqdm(selected_records, desc="Preparing train_pix2pix pairs")):
    pair_id = f"pair_{idx:05d}_{slugify(record.get('generator_model', 'unknown'))}"
    input_path = PAIR_DATASET_DIR / f"{pair_id}_input.png"
    target_path = PAIR_DATASET_DIR / f"{pair_id}_target.png"
    resize_square(Image.open(record["input_image"]), RESOLUTION).save(input_path)
    resize_square(Image.open(record["edited_image"]), RESOLUTION).save(target_path)
    prepared = {
        "pair_id": pair_id,
        "input_image": str(input_path),
        "target_image": str(target_path),
        "source_input_image": record["input_image"],
        "source_edited_image": record["edited_image"],
        "edit_prompt": record["edit_prompt"],
        "generator_model": record.get("generator_model"),
        "class_name": record.get("class_name"),
    }
    prepared_records.append(prepared)

prepared_metadata_path = PAIR_DATASET_DIR / "metadata.jsonl"
with prepared_metadata_path.open("w", encoding="utf-8") as f:
    for record in prepared_records:
        f.write(json.dumps(record) + "\n")

split_manifest = {
    "run_name": RUN_NAME,
    "seed": SEED,
    "source_metadata": str(DATASET_METADATA_PATH),
    "train_pair_count": len(prepared_records),
    "eval_pair_count": len(eval_records),
    "train_pairs": prepared_records,
    "eval_pairs": [prepared_records[i] for i in range(len(eval_records))],
}
(PAIR_DATASET_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, indent=2), encoding="utf-8")
print(f"Prepared pairs: {len(prepared_records)}")
print(f"Saved prepared metadata: {prepared_metadata_path}")


## Fine-Tune InstructPix2Pix

In [ ]:
from diffusers import StableDiffusionInstructPix2PixPipeline, DDPMScheduler
from accelerate import Accelerator
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


def enable_trusted_legacy_ckpt_loading():
    if getattr(torch.load, "_gadang_trusted_ckpt_patch", False):
        return
    original_torch_load = torch.load
    def trusted_torch_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return original_torch_load(*args, **kwargs)
    trusted_torch_load._gadang_trusted_ckpt_patch = True
    torch.load = trusted_torch_load
    log_event("Enabled trusted legacy .ckpt loading with torch.load(weights_only=False)")


def load_ip2p_pipeline():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    enable_trusted_legacy_ckpt_loading()
    log_event(f"Loading local InstructPix2Pix checkpoint: {INSTRUCT_PIX2PIX_CKPT}")
    return StableDiffusionInstructPix2PixPipeline.from_single_file(
        str(INSTRUCT_PIX2PIX_CKPT),
        torch_dtype=dtype,
        safety_checker=None,
    )


image_transform = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

class Pix2PixPairDataset(Dataset):
    def __init__(self, metadata_path: Path):
        self.records = [json.loads(line) for line in metadata_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        record = self.records[index]
        input_image = Image.open(record["input_image"]).convert("RGB")
        target_image = Image.open(record["target_image"]).convert("RGB")
        return {
            "input_pixel_values": image_transform(input_image),
            "target_pixel_values": image_transform(target_image),
            "prompt": record["edit_prompt"],
            "pair_id": record["pair_id"],
        }

def collate_fn(batch):
    return {
        "input_pixel_values": torch.stack([x["input_pixel_values"] for x in batch]),
        "target_pixel_values": torch.stack([x["target_pixel_values"] for x in batch]),
        "prompt": [x["prompt"] for x in batch],
        "pair_id": [x["pair_id"] for x in batch],
    }


def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


In [ ]:

train_dataset = Pix2PixPairDataset(prepared_metadata_path)
train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
accelerator = Accelerator(gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, mixed_precision=MIXED_PRECISION if torch.cuda.is_available() else "no")
pipe = load_ip2p_pipeline()
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)
vae, text_encoder, tokenizer, unet = pipe.vae, pipe.text_encoder, pipe.tokenizer, pipe.unet
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

def configure_trainable_unet(unet, scope_label: str):
    scope = str(scope_label).lower()
    if scope in {"4", "attn_proj", "small"}:
        mode = "attention_projection_only"
        keywords = ["to_q", "to_k", "to_v", "to_out"]
        for name, param in unet.named_parameters():
            param.requires_grad = any(key in name for key in keywords)
    elif scope in {"8", "attention", "medium"}:
        mode = "all_attention_blocks"
        for name, param in unet.named_parameters():
            param.requires_grad = "attn" in name
    else:
        mode = "full_unet"
        for param in unet.parameters():
            param.requires_grad = True
    return mode

trainable_mode = configure_trainable_unet(unet, DREAMBOOTH_SCOPE_LABEL)

if METHOD == "train_pix2pix_dreambooth":
    # Trainable DreamBooth parameters must stay in fp32 under mixed precision.
    unet.to(dtype=torch.float32)
trainable_params, total_params = count_trainable_parameters(unet)
print(f"Trainable mode: {trainable_mode}")
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
if METHOD == "train_pix2pix_dreambooth":
    print("UNet dtype for train_pix2pix DreamBooth parameters: float32")
optimizer = torch.optim.AdamW([p for p in unet.parameters() if p.requires_grad], lr=LEARNING_RATE)
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)
vae.to(accelerator.device); text_encoder.to(accelerator.device)
vae.eval(); text_encoder.eval()
weight_dtype = torch.float16 if accelerator.mixed_precision == "fp16" else torch.float32
vae.to(dtype=weight_dtype); text_encoder.to(dtype=weight_dtype)

checkpoint_records = []

def save_checkpoint(step: int, label: str):
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        checkpoint_root = CHECKPOINT_DIR / label
        checkpoint_root.mkdir(parents=True, exist_ok=True)
        current_unet = accelerator.unwrap_model(unet)
        pipe.unet = current_unet
        pipe.save_pretrained(checkpoint_root / "pipeline")
        current_unet.save_pretrained(checkpoint_root / "unet")
        checkpoint_records.append({"step": int(step), "label": label, "pipeline_path": str(checkpoint_root / "pipeline"), "unet_path": str(checkpoint_root / "unet")})
        log_event(f"Saved DreamBooth checkpoint {label}: {checkpoint_root}")

log_section("Training start")
log_event(f"Optimizer-step budget: {MAX_TRAIN_STEPS}")
log_event(f"Pairs: {len(train_dataset)}; batch: {TRAIN_BATCH_SIZE}; grad accumulation: {GRADIENT_ACCUMULATION_STEPS}")
loss_history = []
global_step = 0
micro_step = 0
batch_iterator = itertools.cycle(train_dataloader)
unet.train()
progress_bar = tqdm(total=MAX_TRAIN_STEPS, desc=f"{RUN_NAME} optimizer steps", disable=not accelerator.is_local_main_process)
while global_step < MAX_TRAIN_STEPS:
    batch = next(batch_iterator)
    micro_step += 1
    with accelerator.accumulate(unet):
        input_pixels = batch["input_pixel_values"].to(accelerator.device, dtype=weight_dtype)
        target_pixels = batch["target_pixel_values"].to(accelerator.device, dtype=weight_dtype)
        with torch.no_grad():
            input_latents = vae.encode(input_pixels).latent_dist.mode() * vae.config.scaling_factor
            target_latents = vae.encode(target_pixels).latent_dist.sample() * vae.config.scaling_factor
            tokenized = tokenizer(batch["prompt"], padding="max_length", truncation=True, max_length=tokenizer.model_max_length, return_tensors="pt").input_ids.to(accelerator.device)
            encoder_hidden_states = text_encoder(tokenized)[0]
        noise = torch.randn_like(target_latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (target_latents.shape[0],), device=target_latents.device).long()
        noisy_target_latents = noise_scheduler.add_noise(target_latents, noise, timesteps)
        latent_model_input = torch.cat([noisy_target_latents, input_latents], dim=1)
        model_pred = unet(latent_model_input, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
        accelerator.backward(loss)
        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if accelerator.sync_gradients:
        global_step += 1
        loss_value = accelerator.gather(loss.detach()).mean().item()
        loss_history.append({"step": global_step, "micro_step": micro_step, "loss": loss_value})
        progress_bar.update(1); progress_bar.set_postfix(loss=f"{loss_value:.4f}")
        if CHECKPOINT_EVERY > 0 and global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(global_step, f"step_{global_step:04d}")
progress_bar.close()
accelerator.wait_for_everyone()

if accelerator.is_main_process:
    if METHOD == "train_pix2pix_lora":
        final_path = CHECKPOINT_DIR / "final" / "lora_adapter"
        final_path.parent.mkdir(parents=True, exist_ok=True)
        accelerator.unwrap_model(unet).save_pretrained(final_path)
        compatibility_path = CHECKPOINT_DIR / "lora_adapter"
        accelerator.unwrap_model(unet).save_pretrained(compatibility_path)
        if not checkpoint_records or checkpoint_records[-1]["step"] != int(global_step):
            checkpoint_records.append({"step": int(global_step), "label": "final", "adapter_path": str(final_path)})
    else:
        final_root = CHECKPOINT_DIR / "final"
        final_root.mkdir(parents=True, exist_ok=True)
        current_unet = accelerator.unwrap_model(unet)
        pipe.unet = current_unet
        pipe.save_pretrained(final_root / "pipeline")
        current_unet.save_pretrained(final_root / "unet")
        pipe.save_pretrained(CHECKPOINT_DIR / "pipeline")
        current_unet.save_pretrained(CHECKPOINT_DIR / "unet")
        if not checkpoint_records or checkpoint_records[-1]["step"] != int(global_step):
            checkpoint_records.append({"step": int(global_step), "label": "final", "pipeline_path": str(final_root / "pipeline"), "unet_path": str(final_root / "unet")})
    (CHECKPOINT_DIR / "checkpoints_manifest.json").write_text(json.dumps({"run_name": RUN_NAME, "checkpoints": checkpoint_records}, indent=2), encoding="utf-8")

pd.DataFrame(loss_history).to_csv(METRICS_DIR / "training_loss.csv", index=False)
RUN_CONFIG.update({"trainable_mode": trainable_mode, "trainable_params": int(trainable_params), "total_params": int(total_params), "actual_optimizer_steps": int(global_step), "actual_micro_steps": int(micro_step), "checkpoint_count": len(checkpoint_records)})
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
log_section("Training finished")


## Evaluation

In [ ]:
from diffusers import StableDiffusionInstructPix2PixPipeline


def load_base_eval_pipeline():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    enable_trusted_legacy_ckpt_loading()
    pipe = StableDiffusionInstructPix2PixPipeline.from_single_file(str(INSTRUCT_PIX2PIX_CKPT), torch_dtype=dtype, safety_checker=None)
    if torch.cuda.is_available():
        pipe = pipe.to("cuda")
        pipe.enable_attention_slicing()
    return pipe


def load_checkpoints():
    manifest_path = CHECKPOINT_DIR / "checkpoints_manifest.json"
    checkpoints = json.loads(manifest_path.read_text(encoding="utf-8"))["checkpoints"] if manifest_path.exists() else []
    return sorted(checkpoints, key=lambda item: (int(item.get("step", 0)), item.get("label") == "final"))


def load_finetuned_pipeline(checkpoint):
    if METHOD == "train_pix2pix_lora":
        pipe = load_base_eval_pipeline()
        adapter_path = Path(checkpoint["adapter_path"])
        pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_path))
        pipe.unet.eval()
        return pipe
    pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(Path(checkpoint["pipeline_path"]), torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, safety_checker=None, local_files_only=True)
    if torch.cuda.is_available():
        pipe = pipe.to("cuda")
        pipe.enable_attention_slicing()
    return pipe


def pil_load(path):
    return Image.open(path).convert("RGB")

@torch.no_grad()
def run_eval_record(record, base_pipe, tuned_pipe, seed):
    input_image = pil_load(record["input_image"])
    target_image = pil_load(record["target_image"])
    prompt = record["edit_prompt"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    base_gen = torch.Generator(device=device).manual_seed(seed)
    tuned_gen = torch.Generator(device=device).manual_seed(seed)
    base_image = base_pipe(prompt=prompt, image=input_image, num_inference_steps=EVAL_NUM_INFERENCE_STEPS, image_guidance_scale=EVAL_IMAGE_GUIDANCE_SCALE, guidance_scale=EVAL_GUIDANCE_SCALE, generator=base_gen).images[0]
    tuned_image = tuned_pipe(prompt=prompt, image=input_image, num_inference_steps=EVAL_NUM_INFERENCE_STEPS, image_guidance_scale=EVAL_IMAGE_GUIDANCE_SCALE, guidance_scale=EVAL_GUIDANCE_SCALE, generator=tuned_gen).images[0]
    return {"input": input_image, "target": target_image, "base": base_image, "fine_tuned": tuned_image, "prompt": prompt}

if RUN_EVALUATION:
    eval_pairs = prepared_records[:min(EVAL_PAIR_COUNT, len(prepared_records))]
    checkpoints = load_checkpoints()
    log_section("Evaluation start")
    log_event(f"Eval pairs: {len(eval_pairs)}; checkpoints: {len(checkpoints)}")
    base_pipe = load_base_eval_pipeline()
    evaluation_records = []
    for checkpoint in checkpoints:
        checkpoint_label = checkpoint["label"]
        tuned_pipe = load_finetuned_pipeline(checkpoint)
        for idx, record in enumerate(eval_pairs):
            result = run_eval_record(record, base_pipe, tuned_pipe, SEED + idx)
            out_dir = EVALUATION_DIR / "checkpoints" / checkpoint_label / "generated_images" / record["pair_id"]
            out_dir.mkdir(parents=True, exist_ok=True)
            paths = {}
            for key in ["input", "target", "base", "fine_tuned"]:
                out_path = out_dir / f"{key}.png"
                result[key].save(out_path)
                paths[key] = str(out_path)
            evaluation_records.append({"run_name": RUN_NAME, "checkpoint_label": checkpoint_label, "checkpoint_step": checkpoint.get("step"), "pair_id": record["pair_id"], "prompt": result["prompt"], **paths})
        del tuned_pipe
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    del base_pipe
    (EVALUATION_DIR / "evaluation_records.json").write_text(json.dumps(evaluation_records, indent=2), encoding="utf-8")
else:
    evaluation_records = []
    (EVALUATION_DIR / "evaluation_records.json").write_text("[]", encoding="utf-8")


## Final Artifacts

In [ ]:
artifact_log = {
    "run_name": RUN_NAME,
    "method": METHOD,
    "output_dir": str(OUTPUT_DIR),
    "pair_dataset_dir": str(PAIR_DATASET_DIR),
    "prepared_metadata": str(prepared_metadata_path),
    "checkpoints_dir": str(CHECKPOINT_DIR),
    "evaluation_dir": str(EVALUATION_DIR),
    "metrics_dir": str(METRICS_DIR),
    "run_config": str(RUN_CONFIG_PATH),
    "run_output_log": str(RUN_LOG_PATH),
}
ARTIFACT_LOG_PATH.write_text(json.dumps(artifact_log, indent=2), encoding="utf-8")
print(json.dumps(artifact_log, indent=2), flush=True)
log_section("Notebook finished")
